In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
file_path = r"D:\Documents\slide unigap\ML\Dataset\Data\churn_training_set.csv"

df = pd.read_csv(file_path)

print(df.shape)
print(df.columns.tolist())

(25559, 15)
['shop_key', 'snapshot_month', 'tenure_in_months', 'bom_mrr', 'eom_mrr', 'gmv_growth_1m', 'gmv_growth_2m', 'searches_1m', 'searches_2m', 'searches_growth_1m', 'searches_growth_2m', 'churn_next_30d', 'split', 'gmv_1m', 'gmv_2m']


In [3]:
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (17183, 15)
Test: (8376, 15)


In [ ]:
#define target and features
target = "churn_next_30d"

exclude_cols = [
    "shop_key",
    "snapshot_month",
    "split",
    target
]

feature_cols = [
    col for col in train_df.columns
    if col not in exclude_cols
]

X_train = train_df[feature_cols].copy()
y_train = train_df[target].copy()
groups = train_df["shop_key"].copy()

print("Features:", feature_cols)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Unique merchants:", groups.nunique())

Features: ['tenure_in_months', 'bom_mrr', 'eom_mrr', 'gmv_growth_1m', 'gmv_growth_2m', 'searches_1m', 'searches_2m', 'searches_growth_1m', 'searches_growth_2m', 'gmv_1m', 'gmv_2m']
X_train: (17183, 11)
y_train: (17183,)
Unique merchants: 8769


In [5]:
# Features groups
log_cols = [
    "bom_mrr",
    "eom_mrr",
    "gmv_1m",
    "gmv_2m",
    "searches_1m",
    "searches_2m"
]

growth_cols = [
    "gmv_growth_1m",
    "gmv_growth_2m",
    "searches_growth_1m",
    "searches_growth_2m"
]

other_cols = ["tenure_in_months"]

### Model 1: Logistic Regression

In [6]:
# preprocssing pipelines
log_pipeline = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

growth_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

other_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_pipeline, log_cols),
        ("growth", growth_pipeline, growth_cols),
        ("other", other_pipeline, other_cols)
    ]
)

In [7]:
# logistic regression pipeline
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [8]:
# cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
from sklearn.model_selection import cross_validate

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    estimator=lr_pipeline,
    X=X_train,
    y=y_train,
    groups=groups,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

In [10]:
results = pd.DataFrame({
    "Fold": range(1, 6),
    "PR-AUC": cv_results["test_pr_auc"],
    "ROC-AUC": cv_results["test_roc_auc"]
})

print(results.round(4))

print("\nMean scores:")
print(results[["PR-AUC", "ROC-AUC"]].mean().round(4))

print("\nStandard deviation:")
print(results[["PR-AUC", "ROC-AUC"]].std().round(4))

   Fold  PR-AUC  ROC-AUC
0     1  0.0568   0.6438
1     2  0.0729   0.6658
2     3  0.0842   0.7280
3     4  0.0863   0.7359
4     5  0.0817   0.7157

Mean scores:
PR-AUC     0.0764
ROC-AUC    0.6978
dtype: float64

Standard deviation:
PR-AUC     0.0121
ROC-AUC    0.0407
dtype: float64


--> The Logistic Regression baseline achieved a mean PR-AUC of 0.0764 and a mean ROC-AUC of 0.6978 across five-fold Stratified Group Cross-Validation. The PR-AUC was approximately 2.84 times the positive class prevalence (2.69%), indicating that the model captured some predictive signals of merchant churn. However, its overall discrimination performance remains limited, suggesting room for improvement through further model development.

In [11]:
# OOP predicted probabilities
from sklearn.base import clone

oof_pred = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    model = clone(lr_pipeline)

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]

    model.fit(X_fold_train, y_fold_train)

    oof_pred[val_idx] = model.predict_proba(X_fold_val)[:, 1]

    print(f"Fold {fold} completed")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed


In [12]:
print("Number of predictions:", len(oof_pred))
print("Minimum probability:", oof_pred.min())
print("Maximum probability:", oof_pred.max())
print("Missing predictions:", np.isnan(oof_pred).sum())

Number of predictions: 17183
Minimum probability: 0.0
Maximum probability: 0.9920128485813993
Missing predictions: 0


In [13]:
print(pd.Series(oof_pred).describe().round(4))

count    17183.0000
mean         0.4411
std          0.1431
min          0.0000
25%          0.3377
50%          0.4123
75%          0.5172
max          0.9920
dtype: float64


In [14]:
oof_df = pd.DataFrame({
    "actual": y_train.values,
    "predicted_probability": oof_pred
})

print(
    oof_df.groupby("actual")["predicted_probability"]
    .describe()
    .round(4)
)

          count    mean     std    min     25%     50%     75%     max
actual                                                                
0       16721.0  0.4379  0.1406  0.000  0.3368  0.4098  0.5128  0.9920
1         462.0  0.5545  0.1809  0.196  0.4270  0.5292  0.6658  0.9821


In [15]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:
    y_pred = (oof_pred >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, y_pred, zero_division=0),
        "Recall": recall_score(y_train, y_pred, zero_division=0),
        "F1-score": f1_score(y_train, y_pred, zero_division=0),
        "Predicted churn": y_pred.sum()
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 Threshold  Precision  Recall  F1-score  Predicted churn
       0.2     0.0270  0.9978    0.0526            17081
       0.3     0.0294  0.9545    0.0571            14985
       0.4     0.0393  0.7835    0.0749             9204
       0.5     0.0545  0.5779    0.0996             4900
       0.6     0.0712  0.3506    0.1183             2276
       0.7     0.0968  0.2078    0.1320              992


In [16]:
from sklearn.metrics import confusion_matrix, classification_report

threshold = 0.5

y_pred = (oof_pred >= threshold).astype(int)

cm = confusion_matrix(y_train, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_train,
    y_pred,
    target_names=["Non-churn", "Churn"],
    zero_division=0
))

Confusion Matrix:
[[12088  4633]
 [  195   267]]

Classification Report:
              precision    recall  f1-score   support

   Non-churn       0.98      0.72      0.83     16721
       Churn       0.05      0.58      0.10       462

    accuracy                           0.72     17183
   macro avg       0.52      0.65      0.47     17183
weighted avg       0.96      0.72      0.81     17183



In [17]:
from sklearn.base import clone

lr_pipeline_unbalanced = clone(lr_pipeline)

lr_pipeline_unbalanced.set_params(
    model__class_weight=None
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...), ('growth', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0

In [18]:
cv_results_unbalanced = cross_validate(
    estimator=lr_pipeline_unbalanced,
    X=X_train,
    y=y_train,
    groups=groups,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

In [19]:
results_unbalanced = pd.DataFrame({
    "Fold": range(1, 6),
    "PR-AUC": cv_results_unbalanced["test_pr_auc"],
    "ROC-AUC": cv_results_unbalanced["test_roc_auc"]
})

print(results_unbalanced.round(4))

print("\nMean scores:")
print(results_unbalanced[["PR-AUC", "ROC-AUC"]].mean().round(4))

print("\nStandard deviation:")
print(results_unbalanced[["PR-AUC", "ROC-AUC"]].std().round(4))

   Fold  PR-AUC  ROC-AUC
0     1  0.0526   0.6297
1     2  0.0690   0.6540
2     3  0.0827   0.7226
3     4  0.0858   0.7332
4     5  0.0823   0.7102

Mean scores:
PR-AUC     0.0745
ROC-AUC    0.6900
dtype: float64

Standard deviation:
PR-AUC     0.0139
ROC-AUC    0.0455
dtype: float64


In [20]:
comparison = pd.DataFrame({
    "Balanced": [
        cv_results["test_pr_auc"].mean(),
        cv_results["test_roc_auc"].mean()
    ],
    "Unbalanced": [
        cv_results_unbalanced["test_pr_auc"].mean(),
        cv_results_unbalanced["test_roc_auc"].mean()
    ]
}, index=["PR-AUC", "ROC-AUC"])

print(comparison.round(4))

         Balanced  Unbalanced
PR-AUC     0.0764      0.0745
ROC-AUC    0.6978      0.6900


### Model 2: Random Forest

In [21]:
from sklearn.ensemble import RandomForestClassifier

In [22]:
# preprocessing pipeline
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "growth",
            SimpleImputer(strategy="median", add_indicator=True),
            growth_cols
        ),
        (
            "other",
            "passthrough",
            log_cols + other_cols
        )
    ]
)

In [ ]:
# model pipeline
rf_pipeline = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=1
    ))
])

In [24]:
# cross-validation
cv_results_rf = cross_validate(
    estimator=rf_pipeline,
    X=X_train,
    y=y_train,
    groups=groups,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

In [25]:
results_rf = pd.DataFrame({
    "Fold": range(1, 6),
    "PR-AUC": cv_results_rf["test_pr_auc"],
    "ROC-AUC": cv_results_rf["test_roc_auc"]
})

print(results_rf.round(4))

print("\nMean scores:")
print(results_rf[["PR-AUC", "ROC-AUC"]].mean().round(4))

print("\nStandard deviation:")
print(results_rf[["PR-AUC", "ROC-AUC"]].std().round(4))

   Fold  PR-AUC  ROC-AUC
0     1  0.0601   0.6740
1     2  0.0755   0.6949
2     3  0.1446   0.7481
3     4  0.0783   0.7423
4     5  0.1031   0.7288

Mean scores:
PR-AUC     0.0923
ROC-AUC    0.7176
dtype: float64

Standard deviation:
PR-AUC     0.033
ROC-AUC    0.032
dtype: float64


In [26]:
fold_stats = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    fold_stats.append({
        "Fold": fold,
        "Train rows": len(train_idx),
        "Validation rows": len(val_idx),
        "Validation merchants": groups.iloc[val_idx].nunique(),
        "Churn cases": y_train.iloc[val_idx].sum(),
        "Churn rate (%)": y_train.iloc[val_idx].mean() * 100
    })

fold_stats_df = pd.DataFrame(fold_stats)

print(fold_stats_df.round(2).to_string(index=False))

 Fold  Train rows  Validation rows  Validation merchants  Churn cases  Churn rate (%)
    1       13746             3437                  1754           92            2.68
    2       13747             3436                  1753           92            2.68
    3       13746             3437                  1754           93            2.71
    4       13746             3437                  1755           93            2.71
    5       13747             3436                  1753           92            2.68


In [27]:
# tạo oop predictions for random forest model
oof_pred_rf = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    model = clone(rf_pipeline)

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]

    model.fit(X_fold_train, y_fold_train)

    oof_pred_rf[val_idx] = model.predict_proba(X_fold_val)[:, 1]

    print(f"Fold {fold} completed")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed


In [28]:
print("Number of predictions:", len(oof_pred_rf))
print("Minimum probability:", oof_pred_rf.min())
print("Maximum probability:", oof_pred_rf.max())
print("Missing predictions:", np.isnan(oof_pred_rf).sum())

Number of predictions: 17183
Minimum probability: 0.0
Maximum probability: 0.9303568075525122
Missing predictions: 0


In [29]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

for threshold in thresholds:
    y_pred_rf = (oof_pred_rf >= threshold).astype(int)

    precision = precision_score(y_train, y_pred_rf, zero_division=0)
    recall = recall_score(y_train, y_pred_rf, zero_division=0)
    f1 = f1_score(y_train, y_pred_rf, zero_division=0)
    predicted_churn = y_pred_rf.sum()

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f} | "
        f"Predicted churn: {predicted_churn}"
    )

Threshold: 0.2 | Precision: 0.0808 | Recall: 0.4221 | F1: 0.1357 | Predicted churn: 2413
Threshold: 0.3 | Precision: 0.1023 | Recall: 0.2857 | F1: 0.1507 | Predicted churn: 1290
Threshold: 0.4 | Precision: 0.1223 | Recall: 0.1861 | F1: 0.1476 | Predicted churn: 703
Threshold: 0.5 | Precision: 0.1260 | Recall: 0.1061 | F1: 0.1152 | Predicted churn: 389
Threshold: 0.6 | Precision: 0.1397 | Recall: 0.0541 | F1: 0.0780 | Predicted churn: 179
Threshold: 0.7 | Precision: 0.1744 | Recall: 0.0325 | F1: 0.0547 | Predicted churn: 86


In [30]:
from sklearn.metrics import average_precision_score, roc_auc_score

models_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "PR-AUC": [
        average_precision_score(y_train, oof_pred),
        average_precision_score(y_train, oof_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_train, oof_pred),
        roc_auc_score(y_train, oof_pred_rf)
    ]
})

models_comparison

,Model,PR-AUC,ROC-AUC
0,Logistic Regression,0.071148,0.695960
1,Random Forest,0.080446,0.716701


In [31]:
# feature importance
from sklearn.inspection import permutation_importance
from sklearn.base import clone

importance_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    model = clone(rf_pipeline)

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Train model
    model.fit(X_fold_train, y_fold_train)

    # Calculate permutation importance on validation set
    result = permutation_importance(
        estimator=model,
        X=X_fold_val,
        y=y_fold_val,
        scoring="average_precision",
        n_repeats=5,
        random_state=42,
        n_jobs=1
    )

    fold_importance = pd.DataFrame({
        "feature": X_train.columns,
        "importance": result.importances_mean,
        "fold": fold
    })

    importance_results.append(fold_importance)

    print(f"Fold {fold} completed")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed


In [32]:
importance_df = pd.concat(
    importance_results,
    ignore_index=True
)

feature_importance = (
    importance_df
    .groupby("feature")["importance"]
    .agg(["mean", "std"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

feature_importance

,feature,mean,std
0,searches_growth_1m,0.019845,0.006259
1,tenure_in_months,0.013112,0.022015
2,bom_mrr,0.007186,0.013200
3,eom_mrr,0.006425,0.015640
4,gmv_growth_1m,0.005313,0.007578
5,searches_growth_2m,0.003645,0.003214
6,searches_2m,0.002829,0.021560
7,searches_1m,0.001920,0.015529
8,gmv_growth_2m,-0.002662,0.004694
9,gmv_1m,-0.006778,0.018925


--> gmv_growth_1m and tenure_in_months are affective the most.

In [ ]:
# tạo bảng oop evaluation
eval_df = train_df.loc[
    X_train.index,
    ["shop_key", "snapshot_month"]
].copy()

eval_df["y_true"] = y_train.values
eval_df["lr_score"] = oof_pred
eval_df["rf_score"] = oof_pred_rf

eval_df.head()

,shop_key,snapshot_month,y_true,lr_score,rf_score
0,cus_02z6hpfbjryw7a,2026-06-01,0,0.963538,0.430312
1,cus_02z6hpfbjryw7a,2026-07-01,0,0.755920,0.322500
3,cus_0ay2nl5xmobqhj,2026-06-01,0,0.838379,0.424459
4,cus_0ay2nl5xmobqhj,2026-07-01,0,0.837019,0.357501
5,cus_2dl9sinz9re2oi,2026-06-01,0,0.774230,0.214728


In [34]:
top_k_values = [50, 100, 200]

results = []

for fold, (_, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    fold_df = eval_df.iloc[val_idx].copy()

    # Chỉ giữ snapshot mới nhất của mỗi merchant
    fold_df = (
        fold_df
        .sort_values("snapshot_month")
        .drop_duplicates("shop_key", keep="last")
    )

    total_churn = fold_df["y_true"].sum()

    for model_name, score_col in [
        ("Logistic Regression", "lr_score"),
        ("Random Forest", "rf_score")
    ]:
        ranked_df = fold_df.sort_values(
            score_col,
            ascending=False
        )

        for k in top_k_values:
            top_k = ranked_df.head(k)

            true_positive = top_k["y_true"].sum()

            results.append({
                "fold": fold,
                "model": model_name,
                "k": k,
                "precision_at_k": true_positive / len(top_k),
                "recall_at_k": true_positive / total_churn
            })

top_k_df = pd.DataFrame(results)

In [35]:
top_k_summary = (
    top_k_df
    .groupby(["model", "k"])
    .agg(
        precision_mean=("precision_at_k", "mean"),
        precision_std=("precision_at_k", "std"),
        recall_mean=("recall_at_k", "mean"),
        recall_std=("recall_at_k", "std")
    )
    .reset_index()
)

top_k_summary

,model,k,precision_mean,precision_std,recall_mean,recall_std
0,Logistic Regression,50,0.232,0.033466,0.125526,0.018001
1,Logistic Regression,100,0.180,0.047434,0.194624,0.050470
2,Logistic Regression,200,0.144,0.023292,0.311547,0.049260
3,Random Forest,50,0.216,0.055498,0.116854,0.029742
4,Random Forest,100,0.208,0.038987,0.225058,0.042003
5,Random Forest,200,0.164,0.027249,0.354885,0.058437


### Model 3: HistGradientBoostingClassifier

In [ ]:
# preprocessing
hgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "growth",
            SimpleImputer(strategy="median", add_indicator=True),
            growth_cols
        ),
        (
            "other",
            "passthrough",
            log_cols + other_cols
        )
    ]
)

In [39]:
# model
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_pipeline = Pipeline([
    ("preprocessor", hgb_preprocessor),
    ("model", HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=42
    ))
])

In [40]:
hgb_cv_results = cross_validate(
    estimator=hgb_pipeline,
    X=X_train,
    y=y_train,
    groups=groups,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

print("PR-AUC:", hgb_cv_results["test_pr_auc"])
print("Mean PR-AUC:", hgb_cv_results["test_pr_auc"].mean())
print("Std PR-AUC:", hgb_cv_results["test_pr_auc"].std())

print("ROC-AUC:", hgb_cv_results["test_roc_auc"])
print("Mean ROC-AUC:", hgb_cv_results["test_roc_auc"].mean())
print("Std ROC-AUC:", hgb_cv_results["test_roc_auc"].std())

PR-AUC: [0.0824329  0.07830161 0.09662268 0.07680563 0.16135425]
Mean PR-AUC: 0.09910341089028989
Std PR-AUC: 0.0319033939968013
ROC-AUC: [0.66835153 0.68043348 0.74429889 0.72750424 0.72335916]
Mean ROC-AUC: 0.7087894619706764
Std ROC-AUC: 0.029198378578867853


In [41]:
oof_pred_hgb = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    model = clone(hgb_pipeline)

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]

    model.fit(X_fold_train, y_fold_train)

    oof_pred_hgb[val_idx] = model.predict_proba(X_fold_val)[:, 1]

    print(f"Fold {fold} completed")
    print("Number of predictions:", len(oof_pred_hgb))
print("Minimum probability:", oof_pred_hgb.min())
print("Maximum probability:", oof_pred_hgb.max())
print("Missing predictions:", np.isnan(oof_pred_hgb).sum())

Fold 1 completed
Number of predictions: 17183
Fold 2 completed
Number of predictions: 17183
Fold 3 completed
Number of predictions: 17183
Fold 4 completed
Number of predictions: 17183
Fold 5 completed
Number of predictions: 17183
Minimum probability: 0.022739005221249307
Maximum probability: 0.8872606234427977
Missing predictions: 0


In [42]:
eval_df["hgb_score"] = oof_pred_hgb

results = []

for fold, (_, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    fold_df = eval_df.iloc[val_idx].copy()

    fold_df = (
        fold_df
        .sort_values("snapshot_month")
        .drop_duplicates("shop_key", keep="last")
    )

    total_churn = fold_df["y_true"].sum()

    for model_name, score_col in [
        ("Logistic Regression", "lr_score"),
        ("Random Forest", "rf_score"),
        ("HistGradientBoosting", "hgb_score")
    ]:
        ranked_df = fold_df.sort_values(
            score_col,
            ascending=False
        )

        for k in [50, 100, 200]:
            top_k = ranked_df.head(k)
            true_positive = top_k["y_true"].sum()

            results.append({
                "fold": fold,
                "model": model_name,
                "k": k,
                "precision_at_k": true_positive / len(top_k),
                "recall_at_k": true_positive / total_churn
            })

top_k_df = pd.DataFrame(results)

top_k_summary = (
    top_k_df
    .groupby(["model", "k"])
    .agg(
        precision_mean=("precision_at_k", "mean"),
        precision_std=("precision_at_k", "std"),
        recall_mean=("recall_at_k", "mean"),
        recall_std=("recall_at_k", "std")
    )
    .reset_index()
)

top_k_summary

,model,k,precision_mean,precision_std,recall_mean,recall_std
0,HistGradientBoosting,50,0.312,0.087864,0.168887,0.047890
1,HistGradientBoosting,100,0.222,0.049699,0.240346,0.054442
2,HistGradientBoosting,200,0.156,0.021909,0.337681,0.047699
3,Logistic Regression,50,0.232,0.033466,0.125526,0.018001
4,Logistic Regression,100,0.180,0.047434,0.194624,0.050470
5,Logistic Regression,200,0.144,0.023292,0.311547,0.049260
6,Random Forest,50,0.216,0.055498,0.116854,0.029742
7,Random Forest,100,0.208,0.038987,0.225058,0.042003
8,Random Forest,200,0.164,0.027249,0.354885,0.058437


--> HistGradientBoosting - final model

In [43]:
# Clone pipeline đã chọn
final_model = clone(hgb_pipeline)

# Train trên toàn bộ training set
final_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['tenure_in_months','bom_mrr','eom_mrr',...,'searches_growth_2m','gmv_1m', 'gmv_2m']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('growth', ...), ('other', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pa

In [44]:
# Tạo X_test và y_test
X_test = test_df[feature_cols].copy()
y_test = test_df["churn_next_30d"].copy()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("Churn cases:", y_test.sum())
print("Churn rate:", y_test.mean())

X_test shape: (8376, 11)
y_test shape: (8376,)
Churn cases: 205
Churn rate: 0.02447468958930277


In [45]:
y_test_proba = final_model.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(y_test_proba))
print("Minimum probability:", y_test_proba.min())
print("Maximum probability:", y_test_proba.max())
print("Missing predictions:", np.isnan(y_test_proba).sum())

Number of predictions: 8376
Minimum probability: 0.07159199126240318
Maximum probability: 0.8805699977872561
Missing predictions: 0


In [46]:
from sklearn.metrics import average_precision_score, roc_auc_score

test_pr_auc = average_precision_score(y_test, y_test_proba)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

print(f"Test PR-AUC: {test_pr_auc:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

Test PR-AUC: 0.0848
Test ROC-AUC: 0.7134


In [47]:
# evaluate top k in test set
test_eval_df = test_df[
    ["shop_key", "snapshot_month"]
].copy()

test_eval_df["y_true"] = y_test.values
test_eval_df["churn_score"] = y_test_proba

print("Total rows:", len(test_eval_df))
print("Unique merchants:", test_eval_df["shop_key"].nunique())

Total rows: 8376
Unique merchants: 8376


In [48]:
top_k_results = []

ranked_test = test_eval_df.sort_values(
    "churn_score",
    ascending=False
)

total_churn = ranked_test["y_true"].sum()

for k in [50, 100, 200]:
    top_k = ranked_test.head(k)

    true_positive = top_k["y_true"].sum()

    top_k_results.append({
        "k": k,
        "actual_churn": int(true_positive),
        "precision_at_k": true_positive / len(top_k),
        "recall_at_k": true_positive / total_churn
    })

test_top_k = pd.DataFrame(top_k_results)

test_top_k

,k,actual_churn,precision_at_k,recall_at_k
0,50,7,0.14,0.034146
1,100,22,0.22,0.107317
2,200,32,0.16,0.156098


In [49]:
# list merchants highest churn probability
high_risk_merchants = (
    test_eval_df
    .sort_values("churn_score", ascending=False)
    .head(100)
    [["shop_key", "churn_score"]]
    .copy()
)

high_risk_merchants.insert(
    0,
    "risk_rank",
    range(1, len(high_risk_merchants) + 1)
)

high_risk_merchants.head(10)

,risk_rank,shop_key,churn_score
1062,1,store-16eyoie1.myshopify.com,0.880570
1459,2,store-1smusnrn.myshopify.com,0.860546
15877,3,store-mco71qe4.myshopify.com,0.860546
10839,4,store-ezi4ji00.myshopify.com,0.859757
8703,5,store-bwv9csjs.myshopify.com,0.859571
4361,6,store-62efqoq6.myshopify.com,0.858538
12291,7,store-h8lrwgj4.myshopify.com,0.856401
13571,8,store-j0cu8eh2.myshopify.com,0.856401
19329,9,store-rbfls3ac.myshopify.com,0.854730
4395,10,store-64bikn5c.myshopify.com,0.854495
